In [1]:
import pandas as pd
import re
from ollama import chat, ChatResponse
from tqdm import tqdm  # 用於顯示進度條

In [2]:
# --- 設定 ---
FILE_PATH = './data/c3_m_test.xlsx'     # 資料集路徑

In [3]:
# --- 函數：呼叫 Ollama (使用官方庫) ---
def query_ollama(prompt, model='gemma3:12b', context_len=1536, think=False):
    try:
        # 使用 ollama.chat 直接呼叫
        response: ChatResponse = chat(
            model=model,
            messages=[
                {'role': 'user', 'content': prompt}
            ],
            # 您可以在此加入 options，例如 temperature
            options={
                'num_ctx': context_len,  # 在這裡指定 Context Length
                'temperature': 0.0  # 讓輸出結果穩定
            },
            think=think  # 設定為 True 以獲取推理欄位
        )
        # 回傳模型的內容
        return response.message.content
    except Exception as e:
        print(f"Error calling Ollama: {e}")
        return ""

In [4]:
# --- 函數：解析模型輸出 ---
def extract_answer(text):
    """
    從模型的回答中提取選項數字 (1, 2, 3, 4)。
    DeepSeek R1 會輸出 <think>...</think> 思考過程，需要過濾。
    """
    if not text:
        return None

    # 1. 移除 <think> 標籤內的內容 (DeepSeek R1 特有)
    # flags=re.DOTALL 確保 . 可以匹配換行符號
    text_cleaned = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    
    # 2. 策略 A: 尋找明確的 "Answer: X" 格式 (Prompt 中有要求)
    match = re.search(r'Answer:\s*([1-4])', text_cleaned, re.IGNORECASE)
    if match:
        return int(match.group(1))
    
    # 3. 策略 B: 如果沒有標準格式，尋找最後出現的數字 1-4
    # 這是一個備用方案
    numbers = re.findall(r'\b([1-4])\b', text_cleaned)
    if numbers:
        return int(numbers[-1])
        
    return None  # 解析失敗

In [5]:
# --- 主程式 ---
# 1. 讀取資料
print(f"正在讀取 {FILE_PATH}...")
try:
    df = pd.read_excel(FILE_PATH)
except Exception as e:
    print(f"讀取檔案失敗: {e}")

# 檢查必要的欄位
required_columns = ['ID', 'Article', 'Question', 'Option1', 'Option2', 'Option3', 'Option4', 'Answer']
if not all(col in df.columns for col in required_columns):
    print(f"錯誤: Excel 檔案缺少必要的欄位。需要: {required_columns}")

正在讀取 ./data/c3_m_test.xlsx...


In [6]:
df

,ID,Article,Question,Option1,Option2,Option3,Option4,Answer
0,1,一个孕妇在路上行走。一个小女孩走过来问她：“阿姨，您的肚子为什么这么大?”孕妇说：“因为肚子...,请选出与试题内容一致的一项。,小女孩怕麻烦,小女孩认为孕妇怕麻烦,孕妇怕麻烦,孕妇很聪明,2
1,2,很多人年轻时觉得钱比健康重要，老了以后才发现自己错了。很多东西都是这样，等到没有了，我们才明...,我们要重视身体健康。,正确,错误,NaN,NaN,1
2,3,一家公司日前推出一款“懒人座椅”，它视重力为无物，利用了永久磁体，令椅子长时间悬在空中，能给...,“懒人座椅”有什么功能?,健身,减肥,飞翔,美体,1
3,4,一家公司日前推出一款“懒人座椅”，它视重力为无物，利用了永久磁体，令椅子长时间悬在空中，能给...,“懒人座椅”的制作方式是：,软件合成,手工制作,批量生产,电脑操控,2
4,5,一家公司日前推出一款“懒人座椅”，它视重力为无物，利用了永久磁体，令椅子长时间悬在空中，能给...,“懒人座椅”有什么特点?,质量轻巧,价格便宜,存放简单,透明化外观,4
...,...,...,...,...,...,...,...,...
1997,1998,一支探险队进人撒哈拉沙漠，在茫茫沙漠里艰难跋涉。烈日如火，队员们个个又渴又累。这时水喝完了，...,故事想告诉我们什么道理?,沙石可以代替水,要成功应该时刻保存信念,探险队不团结,谎言可以帮人成功,2
1998,1999,近日，美国科学家成功地开发出一项新技术，他们制作了一种特殊的帽子，戴上它就可以使人的大脑与网...,请选出与试题内容一致的一项。,帽子可以代替大脑,大脑可以代替天线,大脑可以成为网络,网络可与大脑连接,4
1999,2000,芳芳今27岁了，她想明年去国外工作。她爸爸同意了，但是妈妈不同意。芳芳的妈妈希望她早点儿结婚。,妈妈不希望芳芳：,找工作,早点儿结婚,结婚太晚,NaN,3
2000,2001,美国的一位数学家近日宣布，他已经发明了两个简单的数学公式，能够用来预测一对新婚夫妇能否维持幸...,请选出与试题内容一致的一项。,公式可以维持婚姻,数学家能预测未来,公式非常准确,公式并不简单,3


In [7]:
# --- 主程式 ---
def main():
    correct_count = 0
    total_questions = len(df)
    results = []

    print(f"開始使用 {MODEL_NAME} 處理 {total_questions} 個問題...")

    # 2. 遍歷每一列進行預測
    for index, row in tqdm(df.iterrows(), total=total_questions):
        article = row['Article']
        question = row['Question']
        # 建立選項字典方便取用
        options = {
            1: row['Option1'],
            2: row['Option2'],
            3: row['Option3'],
            4: row['Option4']
        }
        actual_answer = row['Answer']
        #print(actual_answer)

        # 1. 過濾掉 NaN 或空值的選項 (假設 options 是 dict 或 list)
        # 這裡用 str(v) != 'nan' 是為了處理 pandas 的情況
        valid_options = {k: v for k, v in options.items() if v and str(v).lower() != 'nan'}
        
        # 2. 動態生成選項字串
        options_text = "\n".join([f"{k}. {v}" for k, v in valid_options.items()])
        valid_numbers = ", ".join([str(k) for k in valid_options.keys()])

        # 建構 Prompt
        # 這裡的設計是為了讓 DeepSeek R1 能夠先思考再回答，並最後輸出固定格式
        prompt = f"""
Read the following article and answer the multiple-choice question.

Article:
{article}

Question:
{question}

Options:
{options_text}

Instruction:
Analyze the article and select the best option from the available choices listed above ({valid_numbers}). 
If an option is missing or empty, do not select it.

Finally, output the answer in the exact format: "Answer: X", where X is the option number.
Do not output anything else after the answer.
"""
        
        # 呼叫模型
        response_text = query_ollama(prompt, model=MODEL_NAME, think=THINK_MODE)
        
        # 解析答案
        predicted_answer = extract_answer(response_text)
        
        # 比對結果
        is_correct = False
        try:
            # 處理 Excel 讀入的答案可能是浮點數或字串的情況
            actual_answer_int = int(actual_answer)
            if predicted_answer is not None and predicted_answer == actual_answer_int:
                correct_count += 1
                is_correct = True
            else:
                print("Article:", article)
                print("Question:", question)
                print("Option:", "1.", row['Option1'], "2.", row['Option2'], "3.", row['Option3'], "4.", row['Option4'])
                print("Answer:", actual_answer)
                print("Predicted Answer:", predicted_answer)
                print('-'*30)
        except ValueError:
            pass # 忽略無效的標準答案格式

        # 儲存詳細結果
        results.append({
            "ID": row['ID'],
            "Actual": actual_answer,
            "Predicted": predicted_answer,
            "Correct": is_correct,
            "Model_Response": response_text
        })

        #print(predicted_answer)
        if (index+1) % 500 == 0:
            print(correct_count, " / ", (index+1), " = ", 100.0*correct_count/(index+1), "%")
            

    # 3. 計算並輸出正確率
    accuracy = (correct_count / total_questions) * 100 if total_questions > 0 else 0
    print("\n" + "="*30)
    print(f"處理完成！")
    print(f"總題數: {total_questions}")
    print(f"答對題數: {correct_count}")
    print(f"正確率 (Accuracy): {accuracy:.2f}%")
    print("="*30)

    # 4. 將結果存檔
    output_file = "result_analysis.xlsx"
    pd.DataFrame(results).to_excel(output_file, index=False)
    print(f"詳細結果已儲存至 {output_file}")

In [9]:
MODEL_NAME = 'gemma3:12b'  # 模型名稱
THINK_MODE = False
main()

開始使用 gemma3:12b 處理 2002 個問題...


  0%|                                                                               | 3/2002 [00:08<1:11:52,  2.16s/it]

Article: 一家公司日前推出一款“懒人座椅”，它视重力为无物，利用了永久磁体，令椅子长时间悬在空中，能给人一种浮于空中的美妙感觉。还有助于缓解背部肌肉疼痛和头疼等问题，而且能产生磁场，可促进人体健康。
这一座椅用手工制作，采用CAD软件设计。椅子的基座固定在两个导向杆上，也许，只有这样它才不至于飘走。基座有几个类似车轮的东西，使上下两层易于移动。
“懒人椅”不仅给人一种独特之感，而且在视觉上也令人吃惊不已。磁体使用的是干净、明亮的树脂材料，你甚至可以看清楚椅子的每一个零件。另外，每张椅子都有一个席子以及限量版销售证书和序列号。“懒人椅”何时上市销售，售价如何，还是一个谜。
Question: “懒人座椅”有什么功能?
Option: 1. 健身 2. 减肥 3. 飞翔 4. 美体
Answer: 1
Predicted Answer: 3
------------------------------


  0%|▎                                                                                | 7/2002 [00:11<29:38,  1.12it/s]

Article: 由北京人民广播电台教育广播和北京温特莱酒店联合主办的“温特莱杯新歌曲大赛”各项工作已经全面展开。
这次大赛目的在于为广大北京市民提供展示创作、演唱才能的机会，推出新人新作和活跃业余文化生活。参赛歌手的演唱将首先在北京人民广播电台教育广播节目中播出，经听众投票和专家审议，最后的获奖名单将在《北京广播电视报》上公布。
凡北京市民，外地、外籍在京常住者都可报名。报名地点在北京温特莱酒店，报名电话：65863377转168，报名截止日期为11月30日。
Question: 从这篇短文可以知道：
Option: 1. 大赛各项工作已经准备好 2. 大赛由北京广播电台主办 3. 获奖者由听众和专家决定 4. 北京人和外地人都可参加
Answer: 3
Predicted Answer: 4
------------------------------


...

Article: 古人建的桥都是女性的，今人建的桥，似乎都是男性的。且看那西湖上的四大名桥，各自婀娜多姿。先看唐人的断桥，名断实不断，只会在冬日里添上一层薄雪，丝丝粘粘的，从白皑皑的一片中断出一截，乃有断桥之名。再看那一点儿也不长的长桥，贴着水面左弯右拐的，和湖面湿成一片。再从那苏小小病殁的西泠桥畔，漫步到苏大学士的压堤桥前，吟唱起“桥外飞花似郎意，桥边深水似侬情”。可见那古人建的桥，显然是女的。再看苏州网师园中的三步桥，小巧玲珑，笔者只一步就跨过了。而今人建的桥，却又转了个性别似的，都成男的了。就好像笔者五年前去的南京长江大桥，虽已残旧，却依旧让人想起小学课本上毛泽东“一桥飞架南北，天堑变通途”的波澜壮阔。你说那能不是男的吗?再看香港的青马大桥。青马大桥是全球最长的行车及铁路吊桥，青马大桥最美是入夜之际，好像一条金光璀璨的火龙。能说他不是条刚烈的汉子吗?还有悉尼港上的哈伯桥，今年75岁了，依然是世界上同类桥梁中最宽阔的。人会老，桥却不会。桥上一根根又粗又硬又壮又黑又热的钢枝接到一起，每根都是力的表现。
Question: 作者认为桥有男女之分主要是因为：
Option: 1. 桥的风格不同 2. 桥的长短不同 3. 桥的宽窄不同 4. 桥的历史长短不同
Answer: 4
Predicted Answer: 1
------------------------------


100%|█████████████████████████████████████████████████████████████████████████████▉| 2000/2002 [20:59<00:01,  1.60it/s]

Article: 芳芳今27岁了，她想明年去国外工作。她爸爸同意了，但是妈妈不同意。芳芳的妈妈希望她早点儿结婚。
Question: 妈妈不希望芳芳：
Option: 1. 找工作 2. 早点儿结婚 3. 结婚太晚 4. nan
Answer: 3
Predicted Answer: 2
------------------------------
1852  /  2000  =  92.6 %


100%|██████████████████████████████████████████████████████████████████████████████| 2002/2002 [21:00<00:00,  1.59it/s]


處理完成！
總題數: 2002
答對題數: 1854
正確率 (Accuracy): 92.61%
詳細結果已儲存至 result_analysis.xlsx


In [10]:
MODEL_NAME = 'qwen3:8b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()

開始使用 qwen3:8b 處理 2002 個問題...


  0%|▎                                                                                | 7/2002 [00:04<12:19,  2.70it/s]

Article: 由北京人民广播电台教育广播和北京温特莱酒店联合主办的“温特莱杯新歌曲大赛”各项工作已经全面展开。
这次大赛目的在于为广大北京市民提供展示创作、演唱才能的机会，推出新人新作和活跃业余文化生活。参赛歌手的演唱将首先在北京人民广播电台教育广播节目中播出，经听众投票和专家审议，最后的获奖名单将在《北京广播电视报》上公布。
凡北京市民，外地、外籍在京常住者都可报名。报名地点在北京温特莱酒店，报名电话：65863377转168，报名截止日期为11月30日。
Question: 从这篇短文可以知道：
Option: 1. 大赛各项工作已经准备好 2. 大赛由北京广播电台主办 3. 获奖者由听众和专家决定 4. 北京人和外地人都可参加
Answer: 3
Predicted Answer: 4
------------------------------


  1%|▌                                                                               | 13/2002 [00:05<09:54,  3.34it/s]

Article: 中国人似乎历来都很在意左与右的问题。但中国历代到底是尚左还是尚右呢？先秦时代明显是尚左的。《老子》三十一章有言：“吉事尚左，凶事尚右。”河上公注解释说：“左，生位也；右，阴道也。”《老子》同章又云：“君子居则贵左，用兵则贵右。兵者不祥之器，非君子之器。”因此好兵贵右者在老子看来是“乐杀人者”也。总之，左是生位阳位，右则是阴位死位。也因此，古人惟办丧礼时尚右。《礼记•檀弓》篇记孔子有姐之丧，郑注云：“丧尚右，右，阴也；吉尚左，左，阳也。”由此可以看出当时人的观念大体是，左主吉，右主凶。
中国人以后的左右观或许可以从中国历代官制中略见一斑，因为中国的官制常常是同一官职却分左官与右官的。以此观之，唐代宋代都是左比右高一等的。唐宋时的左右仆射、左右丞相、左右丞，皆以左官为上。到了元代蒙古人统治时却反了过来，不但左右丞相和左右丞都是以右官居上，而且元代考科取士也分右榜、左榜，蒙古人列入右榜，汉人则在左榜，左榜自然低于右榜，因为在元代汉人是二等臣民。到了大明朝代，又回到了唐宋时代的尚左传统：明代六部的左右侍郎、左右都御史、左右给事中、左右布政使，都是以左官居上。
近世以来，国民党时期是尚右的，那时说某人思想左倾会有下狱坐牢之祸的。共产党毛泽东时期则是尚左反右的，右倾是要被戴上高帽子游街的。邓小平以后开始反左，知识分子尤其争相标榜自己“反左”最早或最力，尽管如此，反左的人通常仍然很少有人自称“右派”，而是更愿意自称改革派或自由派。
其实左好还是右好，《诗经•小雅》说得最好：“左之左之，君子宜之，右之右之，君子有之。”换言之，该左就左，该右就右，君子无可无不可。
Question: 根据本文第二段我们可以知道：
Option: 1. 唐宋以来左右观是为官制服务的 2. 中国的官制影响了中国的左右观 3. 中国的左右观影响了中国的官制 4. 元代尚右是因为蒙古人崇尚武功
Answer: 4
Predicted Answer: 2
------------------------------


...

Article: 古人建的桥都是女性的，今人建的桥，似乎都是男性的。且看那西湖上的四大名桥，各自婀娜多姿。先看唐人的断桥，名断实不断，只会在冬日里添上一层薄雪，丝丝粘粘的，从白皑皑的一片中断出一截，乃有断桥之名。再看那一点儿也不长的长桥，贴着水面左弯右拐的，和湖面湿成一片。再从那苏小小病殁的西泠桥畔，漫步到苏大学士的压堤桥前，吟唱起“桥外飞花似郎意，桥边深水似侬情”。可见那古人建的桥，显然是女的。再看苏州网师园中的三步桥，小巧玲珑，笔者只一步就跨过了。而今人建的桥，却又转了个性别似的，都成男的了。就好像笔者五年前去的南京长江大桥，虽已残旧，却依旧让人想起小学课本上毛泽东“一桥飞架南北，天堑变通途”的波澜壮阔。你说那能不是男的吗?再看香港的青马大桥。青马大桥是全球最长的行车及铁路吊桥，青马大桥最美是入夜之际，好像一条金光璀璨的火龙。能说他不是条刚烈的汉子吗?还有悉尼港上的哈伯桥，今年75岁了，依然是世界上同类桥梁中最宽阔的。人会老，桥却不会。桥上一根根又粗又硬又壮又黑又热的钢枝接到一起，每根都是力的表现。
Question: 作者认为桥有男女之分主要是因为：
Option: 1. 桥的风格不同 2. 桥的长短不同 3. 桥的宽窄不同 4. 桥的历史长短不同
Answer: 4
Predicted Answer: 1
------------------------------


100%|█████████████████████████████████████████████████████████████████████████████▉| 2000/2002 [09:56<00:00,  3.55it/s]

Article: 芳芳今27岁了，她想明年去国外工作。她爸爸同意了，但是妈妈不同意。芳芳的妈妈希望她早点儿结婚。
Question: 妈妈不希望芳芳：
Option: 1. 找工作 2. 早点儿结婚 3. 结婚太晚 4. nan
Answer: 3
Predicted Answer: 2
------------------------------
1869  /  2000  =  93.45 %


100%|██████████████████████████████████████████████████████████████████████████████| 2002/2002 [09:57<00:00,  3.35it/s]


處理完成！
總題數: 2002
答對題數: 1871
正確率 (Accuracy): 93.46%
詳細結果已儲存至 result_analysis.xlsx


In [8]:
MODEL_NAME = 'qwen2.5:7b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()

開始使用 qwen2.5:7b 處理 2002 個問題...


  0%|                                                                                 | 3/2002 [00:07<59:55,  1.80s/it]

Article: 一家公司日前推出一款“懒人座椅”，它视重力为无物，利用了永久磁体，令椅子长时间悬在空中，能给人一种浮于空中的美妙感觉。还有助于缓解背部肌肉疼痛和头疼等问题，而且能产生磁场，可促进人体健康。
这一座椅用手工制作，采用CAD软件设计。椅子的基座固定在两个导向杆上，也许，只有这样它才不至于飘走。基座有几个类似车轮的东西，使上下两层易于移动。
“懒人椅”不仅给人一种独特之感，而且在视觉上也令人吃惊不已。磁体使用的是干净、明亮的树脂材料，你甚至可以看清楚椅子的每一个零件。另外，每张椅子都有一个席子以及限量版销售证书和序列号。“懒人椅”何时上市销售，售价如何，还是一个谜。
Question: “懒人座椅”有什么功能?
Option: 1. 健身 2. 减肥 3. 飞翔 4. 美体
Answer: 1
Predicted Answer: 3
------------------------------


  0%|▎                                                                                | 7/2002 [00:08<18:38,  1.78it/s]

Article: 由北京人民广播电台教育广播和北京温特莱酒店联合主办的“温特莱杯新歌曲大赛”各项工作已经全面展开。
这次大赛目的在于为广大北京市民提供展示创作、演唱才能的机会，推出新人新作和活跃业余文化生活。参赛歌手的演唱将首先在北京人民广播电台教育广播节目中播出，经听众投票和专家审议，最后的获奖名单将在《北京广播电视报》上公布。
凡北京市民，外地、外籍在京常住者都可报名。报名地点在北京温特莱酒店，报名电话：65863377转168，报名截止日期为11月30日。
Question: 从这篇短文可以知道：
Option: 1. 大赛各项工作已经准备好 2. 大赛由北京广播电台主办 3. 获奖者由听众和专家决定 4. 北京人和外地人都可参加
Answer: 3
Predicted Answer: 1
------------------------------


...

Article: 古人建的桥都是女性的，今人建的桥，似乎都是男性的。且看那西湖上的四大名桥，各自婀娜多姿。先看唐人的断桥，名断实不断，只会在冬日里添上一层薄雪，丝丝粘粘的，从白皑皑的一片中断出一截，乃有断桥之名。再看那一点儿也不长的长桥，贴着水面左弯右拐的，和湖面湿成一片。再从那苏小小病殁的西泠桥畔，漫步到苏大学士的压堤桥前，吟唱起“桥外飞花似郎意，桥边深水似侬情”。可见那古人建的桥，显然是女的。再看苏州网师园中的三步桥，小巧玲珑，笔者只一步就跨过了。而今人建的桥，却又转了个性别似的，都成男的了。就好像笔者五年前去的南京长江大桥，虽已残旧，却依旧让人想起小学课本上毛泽东“一桥飞架南北，天堑变通途”的波澜壮阔。你说那能不是男的吗?再看香港的青马大桥。青马大桥是全球最长的行车及铁路吊桥，青马大桥最美是入夜之际，好像一条金光璀璨的火龙。能说他不是条刚烈的汉子吗?还有悉尼港上的哈伯桥，今年75岁了，依然是世界上同类桥梁中最宽阔的。人会老，桥却不会。桥上一根根又粗又硬又壮又黑又热的钢枝接到一起，每根都是力的表现。
Question: 适合做这段文字标题的是：
Option: 1. 世界上的桥 2. 西湖四大名桥 3. 古桥今桥之大不同 4. 桥男桥女——桥的性格之分析
Answer: 4
Predicted Answer: 3
------------------------------


100%|█████████████████████████████████████████████████████████████████████████████▉| 2000/2002 [10:26<00:00,  3.34it/s]

Article: 芳芳今27岁了，她想明年去国外工作。她爸爸同意了，但是妈妈不同意。芳芳的妈妈希望她早点儿结婚。
Question: 妈妈不希望芳芳：
Option: 1. 找工作 2. 早点儿结婚 3. 结婚太晚 4. nan
Answer: 3
Predicted Answer: 2
------------------------------
1829  /  2000  =  91.45 %


100%|██████████████████████████████████████████████████████████████████████████████| 2002/2002 [10:26<00:00,  3.19it/s]


處理完成！
總題數: 2002
答對題數: 1831
正確率 (Accuracy): 91.46%
詳細結果已儲存至 result_analysis.xlsx


In [11]:
MODEL_NAME = 'qwen3:14b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()

開始使用 qwen3:14b 處理 2002 個問題...


  0%|                                                                                 | 3/2002 [00:03<31:17,  1.06it/s]

Article: 一家公司日前推出一款“懒人座椅”，它视重力为无物，利用了永久磁体，令椅子长时间悬在空中，能给人一种浮于空中的美妙感觉。还有助于缓解背部肌肉疼痛和头疼等问题，而且能产生磁场，可促进人体健康。
这一座椅用手工制作，采用CAD软件设计。椅子的基座固定在两个导向杆上，也许，只有这样它才不至于飘走。基座有几个类似车轮的东西，使上下两层易于移动。
“懒人椅”不仅给人一种独特之感，而且在视觉上也令人吃惊不已。磁体使用的是干净、明亮的树脂材料，你甚至可以看清楚椅子的每一个零件。另外，每张椅子都有一个席子以及限量版销售证书和序列号。“懒人椅”何时上市销售，售价如何，还是一个谜。
Question: “懒人座椅”有什么功能?
Option: 1. 健身 2. 减肥 3. 飞翔 4. 美体
Answer: 1
Predicted Answer: 3
------------------------------


  0%|▎                                                                                | 7/2002 [00:04<13:51,  2.40it/s]

Article: 由北京人民广播电台教育广播和北京温特莱酒店联合主办的“温特莱杯新歌曲大赛”各项工作已经全面展开。
这次大赛目的在于为广大北京市民提供展示创作、演唱才能的机会，推出新人新作和活跃业余文化生活。参赛歌手的演唱将首先在北京人民广播电台教育广播节目中播出，经听众投票和专家审议，最后的获奖名单将在《北京广播电视报》上公布。
凡北京市民，外地、外籍在京常住者都可报名。报名地点在北京温特莱酒店，报名电话：65863377转168，报名截止日期为11月30日。
Question: 从这篇短文可以知道：
Option: 1. 大赛各项工作已经准备好 2. 大赛由北京广播电台主办 3. 获奖者由听众和专家决定 4. 北京人和外地人都可参加
Answer: 3
Predicted Answer: 4
------------------------------


...

Article: 古人建的桥都是女性的，今人建的桥，似乎都是男性的。且看那西湖上的四大名桥，各自婀娜多姿。先看唐人的断桥，名断实不断，只会在冬日里添上一层薄雪，丝丝粘粘的，从白皑皑的一片中断出一截，乃有断桥之名。再看那一点儿也不长的长桥，贴着水面左弯右拐的，和湖面湿成一片。再从那苏小小病殁的西泠桥畔，漫步到苏大学士的压堤桥前，吟唱起“桥外飞花似郎意，桥边深水似侬情”。可见那古人建的桥，显然是女的。再看苏州网师园中的三步桥，小巧玲珑，笔者只一步就跨过了。而今人建的桥，却又转了个性别似的，都成男的了。就好像笔者五年前去的南京长江大桥，虽已残旧，却依旧让人想起小学课本上毛泽东“一桥飞架南北，天堑变通途”的波澜壮阔。你说那能不是男的吗?再看香港的青马大桥。青马大桥是全球最长的行车及铁路吊桥，青马大桥最美是入夜之际，好像一条金光璀璨的火龙。能说他不是条刚烈的汉子吗?还有悉尼港上的哈伯桥，今年75岁了，依然是世界上同类桥梁中最宽阔的。人会老，桥却不会。桥上一根根又粗又硬又壮又黑又热的钢枝接到一起，每根都是力的表现。
Question: 作者认为桥有男女之分主要是因为：
Option: 1. 桥的风格不同 2. 桥的长短不同 3. 桥的宽窄不同 4. 桥的历史长短不同
Answer: 4
Predicted Answer: 1
------------------------------


100%|█████████████████████████████████████████████████████████████████████████████▉| 2000/2002 [11:58<00:00,  2.55it/s]

1895  /  2000  =  94.75 %


100%|██████████████████████████████████████████████████████████████████████████████| 2002/2002 [11:59<00:00,  2.78it/s]



處理完成！
總題數: 2002
答對題數: 1897
正確率 (Accuracy): 94.76%
詳細結果已儲存至 result_analysis.xlsx


In [13]:
MODEL_NAME = 'deepseek-r1:14b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()

開始使用 deepseek-r1:14b 處理 2002 個問題...


  0%|▎                                                                                | 7/2002 [00:14<46:25,  1.40s/it]

Article: 由北京人民广播电台教育广播和北京温特莱酒店联合主办的“温特莱杯新歌曲大赛”各项工作已经全面展开。
这次大赛目的在于为广大北京市民提供展示创作、演唱才能的机会，推出新人新作和活跃业余文化生活。参赛歌手的演唱将首先在北京人民广播电台教育广播节目中播出，经听众投票和专家审议，最后的获奖名单将在《北京广播电视报》上公布。
凡北京市民，外地、外籍在京常住者都可报名。报名地点在北京温特莱酒店，报名电话：65863377转168，报名截止日期为11月30日。
Question: 从这篇短文可以知道：
Option: 1. 大赛各项工作已经准备好 2. 大赛由北京广播电台主办 3. 获奖者由听众和专家决定 4. 北京人和外地人都可参加
Answer: 3
Predicted Answer: 1
------------------------------


  1%|▌                                                                               | 13/2002 [00:24<46:59,  1.42s/it]

Article: 中国人似乎历来都很在意左与右的问题。但中国历代到底是尚左还是尚右呢？先秦时代明显是尚左的。《老子》三十一章有言：“吉事尚左，凶事尚右。”河上公注解释说：“左，生位也；右，阴道也。”《老子》同章又云：“君子居则贵左，用兵则贵右。兵者不祥之器，非君子之器。”因此好兵贵右者在老子看来是“乐杀人者”也。总之，左是生位阳位，右则是阴位死位。也因此，古人惟办丧礼时尚右。《礼记•檀弓》篇记孔子有姐之丧，郑注云：“丧尚右，右，阴也；吉尚左，左，阳也。”由此可以看出当时人的观念大体是，左主吉，右主凶。
中国人以后的左右观或许可以从中国历代官制中略见一斑，因为中国的官制常常是同一官职却分左官与右官的。以此观之，唐代宋代都是左比右高一等的。唐宋时的左右仆射、左右丞相、左右丞，皆以左官为上。到了元代蒙古人统治时却反了过来，不但左右丞相和左右丞都是以右官居上，而且元代考科取士也分右榜、左榜，蒙古人列入右榜，汉人则在左榜，左榜自然低于右榜，因为在元代汉人是二等臣民。到了大明朝代，又回到了唐宋时代的尚左传统：明代六部的左右侍郎、左右都御史、左右给事中、左右布政使，都是以左官居上。
近世以来，国民党时期是尚右的，那时说某人思想左倾会有下狱坐牢之祸的。共产党毛泽东时期则是尚左反右的，右倾是要被戴上高帽子游街的。邓小平以后开始反左，知识分子尤其争相标榜自己“反左”最早或最力，尽管如此，反左的人通常仍然很少有人自称“右派”，而是更愿意自称改革派或自由派。
其实左好还是右好，《诗经•小雅》说得最好：“左之左之，君子宜之，右之右之，君子有之。”换言之，该左就左，该右就右，君子无可无不可。
Question: 根据本文第二段我们可以知道：
Option: 1. 唐宋以来左右观是为官制服务的 2. 中国的官制影响了中国的左右观 3. 中国的左右观影响了中国的官制 4. 元代尚右是因为蒙古人崇尚武功
Answer: 4
Predicted Answer: 2
------------------------------


...

Article: 小王和他女朋友的关系非常好，两个人经常一起吃饭、唱歌、看电影。我觉得他们结婚后一定会过得很幸福。
Question: 根据这段话，可以知道小王：
Option: 1. 没结婚 2. 想去唱歌 3. 喜欢孩子 4. nan
Answer: 1
Predicted Answer: None
------------------------------


100%|█████████████████████████████████████████████████████████████████████████████▉| 2000/2002 [52:04<00:03,  1.84s/it]

Article: 芳芳今27岁了，她想明年去国外工作。她爸爸同意了，但是妈妈不同意。芳芳的妈妈希望她早点儿结婚。
Question: 妈妈不希望芳芳：
Option: 1. 找工作 2. 早点儿结婚 3. 结婚太晚 4. nan
Answer: 3
Predicted Answer: 1
------------------------------
1850  /  2000  =  92.5 %


100%|██████████████████████████████████████████████████████████████████████████████| 2002/2002 [52:06<00:00,  1.56s/it]


處理完成！
總題數: 2002
答對題數: 1852
正確率 (Accuracy): 92.51%
詳細結果已儲存至 result_analysis.xlsx
